In [1]:
import sys
print(sys.executable)

c:\Desktop copy\Legal-Rag-Assistant\venv\Scripts\python.exe


In [2]:
import chromadb
print(chromadb.__version__)

1.5.9


In [3]:
import pandas as pd

chunks_df = pd.read_csv("../data/processed/legal_chunks.csv")

print(chunks_df.shape)

(88437, 4)


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Model Loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Desktop copy\Legal-Rag-Assistant\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tanis\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded


In [5]:
import chromadb

client = chromadb.PersistentClient(
    path="../chroma_db"
)

print("Connected")

Connected


In [6]:
collection = client.get_or_create_collection(
    name="legal_cases"
)

print(collection.count())

0


In [7]:
ids = chunks_df["chunk_id"].astype(str).tolist()

print(ids[:5])

['Case1_0', 'Case2_0', 'Case3_0', 'Case4_0', 'Case5_0']


In [8]:
embeddings = model.encode(
    chunks_df["chunk_text"].tolist(),
    batch_size=64,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/1382 [00:00<?, ?it/s]

(88437, 384)


In [9]:

embeddings_list = embeddings.tolist()

print(type(embeddings_list))
print(len(embeddings_list))

<class 'list'>
88437


In [10]:
metadatas = []

for _, row in chunks_df.iterrows():
    metadatas.append({
        "case_id": str(row["case_id"]),
        "case_outcome": str(row["case_outcome"])
    })

print(len(metadatas))
print(metadatas[0])

88437
{'case_id': 'Case1', 'case_outcome': 'cited'}


In [11]:
batch_size = 1000

for i in range(0, len(ids), batch_size):

    collection.add(
        ids=ids[i:i+batch_size],
        embeddings=embeddings_list[i:i+batch_size],
        documents=chunks_df["chunk_text"].tolist()[i:i+batch_size],
        metadatas=metadatas[i:i+batch_size]
    )

    print(
        f"Inserted {min(i+batch_size, len(ids))}/{len(ids)}"
    )

Inserted 1000/88437
Inserted 2000/88437
Inserted 3000/88437
Inserted 4000/88437
Inserted 5000/88437
Inserted 6000/88437
Inserted 7000/88437
Inserted 8000/88437
Inserted 9000/88437
Inserted 10000/88437
Inserted 11000/88437
Inserted 12000/88437
Inserted 13000/88437
Inserted 14000/88437
Inserted 15000/88437
Inserted 16000/88437
Inserted 17000/88437
Inserted 18000/88437
Inserted 19000/88437
Inserted 20000/88437
Inserted 21000/88437
Inserted 22000/88437
Inserted 23000/88437
Inserted 24000/88437
Inserted 25000/88437
Inserted 26000/88437
Inserted 27000/88437
Inserted 28000/88437
Inserted 29000/88437
Inserted 30000/88437
Inserted 31000/88437
Inserted 32000/88437
Inserted 33000/88437
Inserted 34000/88437
Inserted 35000/88437
Inserted 36000/88437
Inserted 37000/88437
Inserted 38000/88437
Inserted 39000/88437
Inserted 40000/88437
Inserted 41000/88437
Inserted 42000/88437
Inserted 43000/88437
Inserted 44000/88437
Inserted 45000/88437
Inserted 46000/88437
Inserted 47000/88437
Inserted 48000/88437
I

In [12]:
print(collection.count())

88437


In [13]:
query = "breach of contract"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(results["documents"][0][0][:500])

contract for breach of condition arises by reason of a failure of performance which has occurred in the past, provided the failure is of sufficient gravity or relates to a sufficiently major term of the contract. The right to rescind a contract in response to repudiation arises, not so much by reason of a failure of performance in the past as by reason of the manifestation of an intention not to perform contractual obligations in the future." See The Contract of Employment (1976) at p 217. The p
